# 🛰️ Geração de Tracks de Satélites GNSS para uma Data Específica

Este notebook demonstra como gerar **tracks** (trajetórias) de satélites GNSS
(GPS, Galileo, GLONASS, BeiDou) para uma data específica.

## O que são Tracks?

Tracks são as **posições 3D** (x, y, z em km) de satélites ao longo do tempo,
calculadas a partir de **TLEs** (Two-Line Elements) usando a biblioteca
[Skyfield](https://rhodesmill.org/skyfield/).

## Fluxo:

1. 📥 **Baixar TLEs** do SpaceTrack (via `SpaceTrackClient`)
2. 💾 **Persistir** no banco PostgreSQL via `SatelliteRepository`
3. 🔍 **Selecionar** o TLE mais próximo da data alvo (via `TLESelector`)
4. 🚀 **Calcular posições** no tempo com `TrackFactory`
5. 📊 **Visualizar** em 3D e em projeção 2D (alt × az)

---

In [8]:
# ============================================================
# 0. Configuração do ambiente
# ============================================================
from __future__ import annotations

import sys
from pathlib import Path
from datetime import datetime, timezone
import nest_asyncio
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from astropy.time import Time

# Garante que o src/ está no path
SRC = Path.cwd().resolve().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ── Core da aplicação ──
from radioskysims.domain.common import TimeRange
from radioskysims.domain.satellite.models import Satellite, OrbitalElement
from radioskysims.domain.tracks.factory import TrackFactory
from radioskysims.domain.geometry import SkyfieldGeometryProvider

from radioskysims.infrastructure.db.engine import Database
from radioskysims.infrastructure.db.satellite.satelliterepository import SatelliteRepository
from radioskysims.infrastructure.external.satellite_downloader import SpaceTrackClient

nest_asyncio.apply()
print("✅ Imports OK")
print(f"RadioSkySim SRC: {SRC}")

✅ Imports OK
RadioSkySim SRC: /media/lbarosi/barosi_portable1/RadioSkySim/media/lbarosi/barosi_portable1/RadioSkySim/src


---
## 1. Configuração da Data Alvo

Escolha uma data para gerar os tracks. O sistema selecionará
automaticamente os TLEs mais próximos desta data.

In [9]:
# ⏱️ Data alvo da simulação
TARGET_DATE = "2026-07-25"  # ISO format
TARGET_EPOCH = datetime.fromisoformat(TARGET_DATE)

# ⏱️ Janela temporal: 24 horas, resolução de 10 minutos
TIME_RANGE = TimeRange.from_iso(
    start=f"{TARGET_DATE}T00:00:00",
    end=f"{TARGET_DATE}T23:50:00",
    resolution=600,  # segundos
)

print(f"📅 Data alvo:     {TARGET_DATE}")
print(f"⏰ Janela:        {TIME_RANGE}")
print(f"📊 Pontos no tempo: {len(TIME_RANGE.value)}")

📅 Data alvo:     2026-07-25
⏰ Janela:        start_jd=2461246.5 end_jd=2461247.4930555555 resolution_sec=600.0
📊 Pontos no tempo: 144


---
## 2. Conectar ao Banco e Carregar TLEs

Aqui conectamos ao banco PostgreSQL e carregamos os satélites
das constelações GNSS (GPS, Galileo, GLONASS, BeiDou).

Se o banco estiver vazio, o sistema automaticamente:
1. Faz download do **catálogo** do Celestrak
2. **Opcionalmente** baixa TLEs recentes do SpaceTrack

In [10]:
# Conecta ao banco
db = Database()

# Instancia o repositório de satélites
repo = SatelliteRepository(
    engine=db,
    tr=TIME_RANGE,
    constellations=["GPS", "GALILEO", "GLONASS", "BEIDOU"],
)

print(f"📊 Catálogo no banco:")


print(f"\n🔢 NORAD IDs carregados: {len(repo.norad_ids)}")

📊 Catálogo no banco:

🔢 NORAD IDs carregados: 169


## 2.1 Verificar se há TLEs disponíveis

Se os satélites não tiverem elementos orbitais (TLEs), tentamos
baixar do SpaceTrack. Isso requer credenciais configuradas
em `settings.toml` ou variáveis de ambiente `ST_USER`/`ST_PWD`.

In [11]:
# Verifica se os satélites têm TLEs
has_elements = repo.verify(epoch=TARGET_EPOCH)
print(f"📡 TLEs disponíveis e recentes: {has_elements}")

if not has_elements:
    print("⚠️  TLEs não encontrados ou desatualizados. Tentando download do SpaceTrack...")
    try:
        # Tenta baixar TLEs do SpaceTrack
        repo.update_orbital_elements_sync(
            norad_ids=repo.norad_ids,
            epoch=TARGET_EPOCH,
        )
        print("✅ Download concluído!")
    except Exception as e:
        print(f"❌ Falha no download: {e}")
        print("   Usaremos dados sintéticos para demonstração.")
else:
    print("✅ TLEs OK!")

📡 TLEs disponíveis e recentes: False
⚠️  TLEs não encontrados ou desatualizados. Tentando download do SpaceTrack...
✅ Download concluído!


---
## 3. Carregar Satélites do Domínio

Convertemos os registros do banco (`SatID` + `OrbitalElement`)
em objetos do domínio `Satellite`, selecionando o TLE mais próximo
da data alvo.

In [ ]:
# Carrega os satélites com elementos do banco
# O repositório já faz a seleção do TLE mais próximo
repo.reset("satellites").reset("norad_ids")
domain_satellites = repo.query_sats(
    norad_ids=repo.norad_ids,
    epoch=TARGET_EPOCH,
)

print(f"🛰️  Satélites carregados: {len(domain_satellites)}")
print()

# Mostra os primeiros de cada constelação
from collections import Counter

const_counts = Counter(s.constellation for s in domain_satellites)
for const, count in const_counts.most_common():
    print(f"   {const}: {count} satélites")

print()
print("📋 Primeiros 5 satélites:")
for s in domain_satellites[:5]:
    epoch_str = s.epoch.strftime("%Y-%m-%d %H:%M")
    print(
        f"   NORAD {s.norad_id:>6d} | {s.name or 'N/A':<20s} | {s.constellation or 'N/A':<10s} | epoch={epoch_str}"
    )

Nenhum norad_id fornecido ou carregado em self.satellites


🛰️  Satélites carregados: 162

   BEIDOU: 48 satélites
   GALILEO: 34 satélites
   GPS: 32 satélites
   GLONASS: 28 satélites
   IRNSS: 5 satélites
   QZS: 4 satélites
   LUCH: 3 satélites
   SES: 2 satélites
   EUTELSAT: 2 satélites
   INMARSAT: 1 satélites
   ASTRA: 1 satélites
   GALAXY: 1 satélites
   NVS: 1 satélites

📋 Primeiros 5 satélites:
   NORAD  26407 | GPS BIIR-5 (PRN 22)  | GPS        | epoch=2026-07-25 04:40
   NORAD  27663 | GPS BIIR-8 (PRN 16)  | GPS        | epoch=2026-07-25 00:18
   NORAD  28190 | GPS BIIR-11 (PRN 19) | GPS        | epoch=2026-07-24 19:45
   NORAD  28474 | GPS BIIR-13 (PRN 02) | GPS        | epoch=2026-07-24 08:27
   NORAD  28874 | GPS BIIRM-1 (PRN 17) | GPS        | epoch=2026-07-25 23:52


## 3.1 Validar consistência temporal

O modelo `Satellite` valida que o TLE selecionado não esteja
a mais de 5 dias da data alvo. Isso garante precisão posicional.

In [ ]:
# Verifica a diferença temporal entre TLEs e data alvo
delta_days = []
for s in domain_satellites[:10]:  # primeiros 10
    d = abs((s.epoch - TARGET_EPOCH).total_seconds()) / 86400
    delta_days.append(d)

if delta_days:
    print(f"📊 Distância média dos TLEs à data alvo: {np.mean(delta_days):.2f} dias")
    print(f"   Máxima: {np.max(delta_days):.2f} dias")
    print(f"   Mínima: {np.min(delta_days):.2f} dias")

📊 Distância média dos TLEs à data alvo: 0.46 dias
   Máxima: 1.22 dias
   Mínima: 0.01 dias


---
## 4. Gerar Tracks (Posições 3D)

A `TrackFactory` calcula as posições (x, y, z em km) de
todos os satélites ao longo do tempo usando Skyfield.

O cálculo é **lazy** (usando Dask) — só executa quando
solicitamos os dados (`compute()`).

In [ ]:
# Monta os blocos de satélites para a TrackFactory
sat_blocks = {
    float(TIME_RANGE.start_jd): {
        "timerange": TIME_RANGE,
        "satellites": domain_satellites,
    }
}

# Instancia a TrackFactory
track_factory = TrackFactory()

# Gera os tracks (lazy!)
tracks = track_factory.build(sat_blocks)

print(f"✅ Tracks gerados!")
print(f"   Dimensões: {dict(tracks.dataset.sizes)}")
print(f"   Satélites: {tracks.satellite_names[:5]}... ({len(tracks.satellite_names)} total)")
print(f"   Tempo:     {tracks.time_range}")

## 4.1 Inspecionar o Dataset

O objeto `Tracks` encapsula um `xr.Dataset` com:
- `positions`: (satellite × time × xyz) em km
- `names`, `constellation`, `tle_line1`, `tle_line2`: metadados

In [ ]:
# Inspeciona o Dataset de tracks
ds = tracks.dataset
print(ds)
print()
print(f"🔍 Dimensões: {dict(ds.sizes)}")
print(f"📊 Variáveis: {list(ds.data_vars)}")
print(f"📍 Coordenadas: {list(ds.coords)}")

In [ ]:
# Computa um pedaço dos dados para inspecionar
# (cuidado: computar tudo pode ser pesado)
sample = ds.isel(satellite=0, time=slice(0, 5))  # primeiro sat, 5 instantes
print("📋 Amostra (primeiro satélite, 5 instantes):")
print(sample["positions"].values)

---
## 5. Calcular Geometria (Alt/Az/Distância)

As posições geocêntricas (x,y,z) não são úteis diretamente —
precisamos convertê-las para coordenadas **topocêntricas**
(altitude, azimute, distância) a partir do observador.

Vamos usar o BINGO como observador de exemplo.

In [ ]:
# Define o observador (BINGO)
from radioskysims.domain.common import Observer

observer = Observer(
    lat=-7.25,
    lon=-35.86,
    height=550.0,
    timeframe=TIME_RANGE,
)

print(f"📍 Observador: lat={observer.lat:.2f}, lon={observer.lon:.2f}, alt={observer.height:.0f}")

In [ ]:
# Calcula a geometria (alt, az, range, theta, ra, dec) para cada satélite
geometry = SkyfieldGeometryProvider(track_factory.skyfield.ts)
geo_ds = geometry.compute(tracks.dataset, observer)

print(f"✅ Geometria calculada!")
print()
print(geo_ds)
print()
print(f"📊 Variáveis:")
for var in geo_ds.data_vars:
    print(
        f"   {var}: shape={geo_ds[var].shape}, unidades={geo_ds[var].attrs.get('units', 'rad/m')}"
    )

---
## 6. Visualização

## 6.1 Tracks 3D (posições geocêntricas)

In [ ]:
# Seleciona algumas constelações para visualizar
gps_mask = ds["constellation"] == "GPS"
galileo_mask = ds["constellation"] == "GALILEO"

gps_positions = ds["positions"].sel(satellite=gps_mask).compute()
galileo_positions = ds["positions"].sel(satellite=galileo_mask).compute()

fig = plt.figure(figsize=(14, 6))

# --- Painel 3D ---
ax1 = fig.add_subplot(121, projection="3d")

for i in range(min(8, gps_positions.sizes["satellite"])):
    traj = gps_positions.isel(satellite=i)
    ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], "b-", alpha=0.5, linewidth=0.8)

for i in range(min(4, galileo_positions.sizes["satellite"])):
    traj = galileo_positions.isel(satellite=i)
    ax1.plot(traj[:, 0], traj[:, 1], traj[:, 2], "r-", alpha=0.5, linewidth=0.8)

# Terra (esfera simplificada)
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, np.pi, 30)
R = 6371  # km
x = R * np.outer(np.cos(u), np.sin(v))
y = R * np.outer(np.sin(u), np.sin(v))
z = R * np.outer(np.ones(np.size(u)), np.cos(v))
ax1.plot_wireframe(x, y, z, color="gray", alpha=0.2, linewidth=0.3)

# Posição do BINGO
lat_r = np.radians(-7.25)
lon_r = np.radians(-35.86)
bingo_x = R * np.cos(lat_r) * np.cos(lon_r)
bingo_y = R * np.cos(lat_r) * np.sin(lon_r)
bingo_z = R * np.sin(lat_r)
ax1.scatter([bingo_x], [bingo_y], [bingo_z], c="k", s=100, marker="^", label="BINGO")

ax1.set_xlabel("X (km)")
ax1.set_ylabel("Y (km)")
ax1.set_zlabel("Z (km)")
ax1.set_title("Tracks 3D - GPS (azul) / Galileo (vermelho)")
ax1.legend()

# --- Painel Alt × Az (diagrama celeste) ---
ax2 = fig.add_subplot(122, projection="polar")


# Converte azimute para convenção polar: 0° = Norte, aumentando CW
def az_to_polar(az_rad):
    """Converte azimute (0=N, CW) para ângulo polar (0=E, CCW)."""
    return (np.pi / 2 - az_rad) % (2 * np.pi)


for i in range(min(8, geo_ds.sizes["satellite"])):
    az = geo_ds["az"].isel(satellite=i).values
    el = geo_ds["el"].isel(satellite=i).values
    # Filtra ângulos válidos (acima do horizonte)
    above_horizon = el > 0
    if not above_horizon.any():
        continue
    theta_polar = az_to_polar(az[above_horizon])
    r_plot = np.pi / 2 - el[above_horizon]  # 0=zenit, pi/2=horizonte
    ax2.plot(theta_polar, r_plot, "-", alpha=0.6, linewidth=0.8)

ax2.set_ylim(0, np.pi / 2)
ax2.set_title("Diagrama Celeste (Alt × Az)")
ax2.set_xlabel("Azimute")
ax2.set_yticks([0, np.pi / 6, np.pi / 3, np.pi / 2])
ax2.set_yticklabels(["90°", "60°", "30°", "0°"])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6.2 Altitude de satélites individuais ao longo do dia

In [ ]:
# Converte tempo para horas do dia
time_hours = (
    Time(geo_ds["time"].values, format="jd").to_datetime()
    - Time(TIME_RANGE.start_jd, format="jd").to_datetime()
)
time_hours = np.array([t.total_seconds() for t in time_hours]) / 3600

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (const, color) in zip(
    axes.flatten(),
    [("GPS", "blue"), ("GALILEO", "red"), ("GLONASS", "green"), ("BEIDOU", "orange")],
):
    # Satélites desta constelação
    mask = geo_ds["satellite"].isin(
        ds["satellite"].where(ds["constellation"] == const, drop=True).values
    )

    for i in range(geo_ds.sizes["satellite"]):
        el = geo_ds["el"].isel(satellite=i).values
        el_deg = np.degrees(el)
        ax.plot(time_hours, el_deg, "-", alpha=0.4, color=color, linewidth=0.6)

    ax.axhline(0, color="gray", linestyle="--", alpha=0.5, label="Horizonte")
    ax.set_xlabel("Horas desde 0h UTC")
    ax.set_ylabel("Elevação (graus)")
    ax.set_title(f"{const} - Elevação ao longo do dia")
    ax.set_ylim(-5, 95)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6.3 Distância dos satélites ao observador

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Plota a distância de cada satélite
for i in range(min(5, geo_ds.sizes["satellite"])):
    r_km = geo_ds["range"].isel(satellite=i).values / 1000
    norad = geo_ds["satellite"].values[i]
    ax.plot(time_hours, r_km, "-", alpha=0.7, linewidth=0.8, label=f"NORAD {norad}")

ax.set_xlabel("Horas desde 0h UTC")
ax.set_ylabel("Distância (km)")
ax.set_title("Distância de satélites GNSS ao BINGO")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

---
## 7. Exportar Tracks

Os tracks podem ser exportados para:
- **HDF5** (.h5) — formato compacto, boa interoperabilidade
- **Zarr** — formato chunked, ideal para datasets grandes
- **NetCDF** — formato padrão em ciências atmosféricas

In [ ]:
# Exportar para HDF5
output_dir = Path.cwd().parent / "data" / "tracks"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / f"gnss_tracks_{TARGET_DATE}.h5"
tracks.to_hdf5(str(output_file))

print(f"✅ Tracks exportados para: {output_file}")
print(f"   Tamanho: {output_file.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# Re-carregar do arquivo HDF5
import xarray as xr

ds_loaded = xr.open_dataset(output_file)
print(f"✅ Dataset carregado de {output_file.name}")
print(ds_loaded)
print()
print(f"📊 Dimensões: {dict(ds_loaded.sizes)}")
print(f"📋 Satélites: {ds_loaded['names'].values.tolist()}")

---
## 8. Resumo e Próximos Passos

### ✅ O que foi feito:

1. **Carregamos TLEs** do banco (com fallback para SpaceTrack)
2. **Selecionamos** o TLE mais próximo da data alvo
3. **Calculamos posições 3D** (x,y,z) em km via Skyfield
4. **Convertemos para geometria observacional** (alt, az, distância)
5. **Visualizamos** em 3D e em projeção celeste

### 🚀 Para continuar:

- Calcular **ângulo boresight** para simular temperatura de antena
- Adicionar **sinais GNSS** (PSDs) para simular potência recebida
- Combinar com **fontes locais** (Sol) para simulação multi-fonte
- Usar o `SimOrchestrator` para uma simulação completa